# FLUX.2 Klein 9B — Garment T-pose Conversion + Gradio API

Auto-generated notebook for recipe: **flux2-tpose**


# FLUX.2 Klein 9B — Garment T-pose Conversion

단일 의류 사진을 **T-pose 정면 이미지**로 변환합니다.

- **Model**: `black-forest-labs/FLUX.2-klein-9B` (9B params, 4-step distilled)
- **Input**: 의류 사진 (평면촬영, 행거샷, 착용샷)
- **Output**: T-pose 의류 이미지 (흰배경, 1024x1024)
- **API**: Gradio 엔드포인트로 외부 호출 가능

---


## A) GPU Check


In [ ]:
#@title A) GPU & VRAM Check { run: "auto" }
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available! Change runtime: Runtime > Change runtime type > GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if vram_gb < 29:
    print("WARNING: VRAM < 29GB. cpu_offload will be used but may be slow.")
elif vram_gb >= 40:
    print(f"OK: {vram_gb:.0f}GB — full GPU mode (no offload needed)")
else:
    print("OK: Sufficient VRAM for FLUX.2 Klein 9B")


## B) Install Dependencies


In [ ]:
#@title B) Install Dependencies { run: "auto" }
import importlib, subprocess, sys, os

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# --- 1. Probe Flux2KleinPipeline ---
_need_restart = False
try:
    from diffusers import Flux2KleinPipeline
    print(f"Flux2KleinPipeline found in diffusers {importlib.metadata.version('diffusers')}")
except ImportError:
    print("Flux2KleinPipeline not in current diffusers — installing git HEAD...")
    pip_install("git+https://github.com/huggingface/diffusers.git")
    _need_restart = True

# --- 2. rembg + gradio + deps ---
for pkg in ["rembg[gpu]", "gradio>=5.0", "accelerate>=1.12.0",
             "sentencepiece", "protobuf", "safetensors"]:
    pip_install(pkg)
print("Additional dependencies installed")

# --- 3. Restart or verify ---
if _need_restart:
    print("\nRestarting runtime to pick up new diffusers...")
    print(">>> After restart, re-run from this cell (B). It will skip install and verify.")
    import signal
    os.kill(os.getpid(), signal.SIGKILL)
else:
    import torch, transformers, accelerate
    print(f"\ntorch={torch.__version__}, CUDA={torch.version.cuda}")
    print(f"transformers={transformers.__version__}")
    print(f"accelerate={accelerate.__version__}")
    print(f"diffusers={importlib.metadata.version('diffusers')}")
    print("\nAll ready.")


## C) HuggingFace Authentication

FLUX.2 Klein 9B는 gated model — HF 토큰이 필요합니다.


In [ ]:
#@title C) HuggingFace Login { run: "auto" }
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Colab secret 'HF_TOKEN'")
except Exception:
    print("HF_TOKEN not found in Colab secrets.")
    print("If download fails, add your token: Colab sidebar > Secrets > HF_TOKEN")
    print("Or run: huggingface_hub.login()")


## D) Load FLUX.2 Klein 9B


In [ ]:
#@title D) Load Model { run: "auto" }
import torch
from diffusers import Flux2KleinPipeline

MODEL_ID = "black-forest-labs/FLUX.2-klein-9B"
DTYPE = torch.bfloat16

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"Loading {MODEL_ID} ...")
pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)

if vram_gb >= 40:
    pipe.to("cuda")
    print(f"Full GPU mode (VRAM: {vram_gb:.0f}GB)")
else:
    pipe.enable_model_cpu_offload()
    print(f"CPU offload mode (VRAM: {vram_gb:.0f}GB)")

vram_after = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded. VRAM used: {vram_after:.2f} GB")
print("Ready for inference.")


## E) Preprocessing — Background Removal


In [ ]:
#@title E) Background Removal + Centering Functions
from rembg import remove
from PIL import Image
import numpy as np
import io

def remove_background(img: Image.Image) -> Image.Image:
    """Remove background using rembg, return RGBA image."""
    img_bytes = io.BytesIO()
    img.save(img_bytes, format="PNG")
    result_bytes = remove(img_bytes.getvalue())
    return Image.open(io.BytesIO(result_bytes)).convert("RGBA")

def center_on_white(rgba_img: Image.Image, target_w: int = 1536, target_h: int = 1024, padding: float = 0.05) -> Image.Image:
    """Center the garment on a white background with padding.
    Default 1536x1024 (landscape) to give horizontal space for T-pose sleeves."""
    alpha = np.array(rgba_img)[:, :, 3]
    rows = np.any(alpha > 10, axis=1)
    cols = np.any(alpha > 10, axis=0)

    if not rows.any() or not cols.any():
        return Image.new("RGB", (target_w, target_h), (255, 255, 255))

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    cropped = rgba_img.crop((cmin, rmin, cmax + 1, rmax + 1))
    cw, ch = cropped.size

    # Scale down to ~60% of canvas height to leave room for sleeves to spread
    pad_px = int(min(target_w, target_h) * padding)
    avail_w = target_w - 2 * pad_px
    avail_h = target_h - 2 * pad_px
    scale = min(avail_w / cw, avail_h / ch) * 0.7  # 70% fill — leave space for sleeve rotation
    new_w = int(cw * scale)
    new_h = int(ch * scale)

    cropped_resized = cropped.resize((new_w, new_h), Image.LANCZOS)

    result = Image.new("RGB", (target_w, target_h), (255, 255, 255))
    offset_x = (target_w - new_w) // 2
    offset_y = (target_h - new_h) // 2
    result.paste(cropped_resized, (offset_x, offset_y), cropped_resized)

    return result

def preprocess_garment(img: Image.Image, target_w: int = 1536, target_h: int = 1024) -> Image.Image:
    """Full preprocessing: remove bg + center on wide canvas."""
    rgba = remove_background(img)
    return center_on_white(rgba, target_w, target_h)

print("Preprocessing functions loaded.")


## F) T-pose Prompt Builder

프롬프트 엔지니어링 전략:
- **Subject + Action + Style + Context** 구조
- 100단어 이내 (Klein 최적)
- T-pose/flat-lay를 맨 앞에 (decoder-only causal attention)
- No negative prompts (FLUX.2 미지원)


In [ ]:
#@title F) Prompt Builder

GARMENT_TYPES = {
    "t-shirt": "short-sleeve t-shirt",
    "hoodie": "hooded sweatshirt",
    "jacket": "zip-up jacket",
    "shirt": "button-down collared shirt",
    "sweater": "knit pullover sweater",
    "coat": "long coat",
    "dress": "dress",
    "polo": "polo shirt",
    "vest": "vest",
    "custom": "",
}

def build_tpose_prompt(garment_type: str = "t-shirt", custom_desc: str = "") -> str:
    """Build the T-pose conversion prompt (edit-instruction style for Klein 9B).
    Key: frame as sleeve ROTATION, not redesign. Preserve original proportions."""
    if garment_type == "custom" and custom_desc:
        garment_desc = custom_desc
    else:
        garment_desc = GARMENT_TYPES.get(garment_type, "garment")

    prompt = (
        f"Edit the input {garment_desc} photo by ONLY repositioning the sleeves. "
        f"Rotate both sleeves outward around the shoulder seams until perfectly horizontal, "
        f"symmetric left and right, creating a flat-lay T-pose layout. "
        f"Do NOT shorten the sleeves. Preserve the original sleeve length from shoulder seam to cuff. "
        f"Keep cuff size, sleeve opening, torso width, and body length the same as the input. "
        f"This is a rotation of the existing sleeves, not a redesign. "
        f"Preserve all garment identity exactly: color, fabric texture, stitching, seams, "
        f"hood shape, pockets, and any graphic or text placement. "
        f"Sleeve ends should reach close to the left and right image margins. "
        f"Garment only, no person, no mannequin, no hanger, clean white background, fully visible."
    )
    return prompt

# Preview
sample_prompt = build_tpose_prompt("t-shirt")
word_count = len(sample_prompt.split())
print(f"Sample prompt ({word_count} words):")
print(sample_prompt)


## G) Single Inference Test

의류 이미지를 업로드하고 T-pose 변환을 테스트합니다.


In [ ]:
#@title G) Upload & Convert to T-pose
import torch
from PIL import Image
from google.colab import files
import time

#@markdown ### Settings
GARMENT_TYPE = "t-shirt"  #@param ["t-shirt", "hoodie", "jacket", "shirt", "sweater", "coat", "dress", "polo", "vest", "custom"]
CUSTOM_DESC = ""  #@param {type:"string"}
GUIDANCE = 4.0  #@param {type:"slider", min:1.0, max:10.0, step:0.5}
NUM_STEPS = 4  #@param {type:"slider", min:4, max:50, step:1}
SEED = 42  #@param {type:"integer"}
OUT_W = 1536  #@param [1024, 1280, 1536] {type:"raw"}
OUT_H = 1024  #@param [768, 1024] {type:"raw"}

# --- Upload ---
print("Upload a garment image:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
garment_img = Image.open(filename).convert("RGB")
print(f"Uploaded: {filename} ({garment_img.size})")

# --- Preprocess (wide canvas for sleeve room) ---
print("Removing background...")
preprocessed = preprocess_garment(garment_img, OUT_W, OUT_H)
display(preprocessed)
print(f"Preprocessed: {preprocessed.size}")

# --- Build prompt ---
prompt = build_tpose_prompt(GARMENT_TYPE, CUSTOM_DESC)
print(f"\nPrompt: {prompt}")

# --- Inference (in-context conditioning, BFL Space pattern) ---
print(f"\nGenerating T-pose ({OUT_W}x{OUT_H}, steps={NUM_STEPS}, guidance={GUIDANCE}, seed={SEED})...")
generator = torch.Generator(device="cuda").manual_seed(SEED)

kwargs = dict(
    prompt=prompt,
    image=[preprocessed],
    height=OUT_H,
    width=OUT_W,
    guidance_scale=GUIDANCE,
    num_inference_steps=NUM_STEPS,
    generator=generator,
)

t0 = time.time()
result = pipe(**kwargs).images[0]
elapsed = time.time() - t0

print(f"Done in {elapsed:.1f}s")
display(result)

# --- Save ---
out_path = f"tpose_{filename}"
result.save(out_path)
print(f"Saved: {out_path}")


## H) Gradio App + API Endpoint

`share=True`로 공개 URL이 생성됩니다.
Gradio Client로 API 호출 가능:

```python
from gradio_client import Client
client = Client("https://xxxxx.gradio.live")
result = client.predict(
    image="garment.jpg",
    garment_type="t-shirt",
    seed=42,
    api_name="/convert"
)
```


In [ ]:
#@title H) Launch Gradio App + API { run: "auto" }
import gradio as gr
import torch
from PIL import Image
import time
import numpy as np

def convert_to_tpose(
    image: Image.Image,
    garment_type: str = "t-shirt",
    custom_desc: str = "",
    guidance: float = 4.0,
    num_steps: int = 4,
    seed: int = 42,
    out_w: int = 1536,
    out_h: int = 1024,
) -> tuple[Image.Image, Image.Image, str]:
    """Convert garment image to T-pose. Returns (preprocessed, result, info)."""
    if image is None:
        raise gr.Error("Please upload a garment image.")

    image = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    image = image.convert("RGB")

    # Preprocess (wide canvas for sleeve room)
    preprocessed = preprocess_garment(image, out_w, out_h)

    # Build prompt
    prompt = build_tpose_prompt(garment_type, custom_desc)

    # Inference (BFL Space pattern)
    generator = torch.Generator(device="cuda").manual_seed(seed)
    kwargs = dict(
        prompt=prompt,
        image=[preprocessed],
        height=out_h,
        width=out_w,
        guidance_scale=guidance,
        num_inference_steps=num_steps,
        generator=generator,
    )
    t0 = time.time()
    result = pipe(**kwargs).images[0]
    elapsed = time.time() - t0

    info = f"Time: {elapsed:.1f}s | {out_w}x{out_h} | Guidance: {guidance} | Steps: {num_steps} | Seed: {seed}"
    return preprocessed, result, info

# --- Build Gradio UI ---
with gr.Blocks(title="Garment T-pose Converter") as demo:
    gr.Markdown("# Garment T-pose Converter\nUpload a garment photo to convert it to T-pose flat-lay format.")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="Input Garment", type="pil")
            garment_type = gr.Dropdown(
                choices=list(GARMENT_TYPES.keys()),
                value="t-shirt",
                label="Garment Type",
            )
            custom_desc = gr.Textbox(label="Custom Description (if type=custom)", visible=True)
            guidance = gr.Slider(1.0, 10.0, value=4.0, step=0.5, label="Guidance Scale")
            num_steps = gr.Slider(4, 50, value=4, step=1, label="Inference Steps")
            seed = gr.Number(value=42, label="Seed", precision=0)
            out_w = gr.Radio([1024, 1280, 1536], value=1536, label="Output Width")
            out_h = gr.Radio([768, 1024], value=1024, label="Output Height")
            btn = gr.Button("Convert to T-pose", variant="primary")

        with gr.Column():
            preprocessed_output = gr.Image(label="Preprocessed (bg removed)")
            result_output = gr.Image(label="T-pose Result")
            info_output = gr.Textbox(label="Info")

    btn.click(
        fn=convert_to_tpose,
        inputs=[input_image, garment_type, custom_desc, guidance, num_steps, seed, out_w, out_h],
        outputs=[preprocessed_output, result_output, info_output],
        api_name="convert",
    )

# --- Launch ---
print("Launching Gradio app with public URL...")
demo.launch(share=True, quiet=False)
